# Granite Embedding Models

In [1]:
import warnings
warnings.filterwarnings('ignore')

## Encode with our model

### Use our models with Sentence Transformers

In [4]:
from sentence_transformers import SentenceTransformer, util

device="cuda"

model_path = "ibm-granite/granite-embedding-english-r2"
# Load the Sentence Transformer model
model = SentenceTransformer(model_path, device=device)

input_queries = [
    ' Who made the song My achy breaky heart? ',
    'summit define'
    ]

input_passages = [
    "Achy Breaky Heart is a country song written by Don Von Tress. Originally titled Don't Tell My Heart and performed by The Marcy Brothers in 1991. ",
    "Definition of summit for English Language Learners. : 1 the highest point of a mountain : the top of a mountain. : 2 the highest level. : 3 a meeting or series of meetings between the leaders of two or more governments."
    ]

# encode queries and passages. The model produces unnormalized vectors. If your task requires normalized embeddings pass normalize_embeddings=True to encode as below.
query_embeddings = model.encode(input_queries)
passage_embeddings = model.encode(input_passages)

# calculate cosine similarity
print(util.cos_sim(query_embeddings, passage_embeddings))

tensor([[0.8945, 0.6078],
        [0.6356, 0.8327]])


### Use our models with Huggingface Transformers

In [7]:
import torch
from transformers import AutoModel, AutoTokenizer

model_path = "ibm-granite/granite-embedding-english-r2"

# Load the model and tokenizer
model = AutoModel.from_pretrained(model_path).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_path)
model.eval()

input_queries = [
    ' Who made the song My achy breaky heart? ',
    'summit define'
    ]

# tokenize inputs
tokenized_queries = tokenizer(input_queries, padding=True, truncation=True, return_tensors='pt').to(device)

# encode queries
with torch.no_grad():
    # Queries
    model_output = model(**tokenized_queries)
    # Perform pooling. granite-embedding-278m-multilingual uses CLS Pooling
    query_embeddings = model_output[0][:, 0]

# normalize the embeddings
query_embeddings = torch.nn.functional.normalize(query_embeddings, dim=1)

In [8]:
query_embeddings

tensor([[ 0.0477,  0.0272,  0.0018,  ..., -0.0736,  0.0055,  0.0233],
        [-0.0167, -0.0218, -0.0187,  ..., -0.0294,  0.0276,  0.0184]],
       device='cuda:0')

## Finetune our model on your own data

In [9]:
from datasets import load_dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.evaluation import TripletEvaluator
from sentence_transformers.losses import CachedMultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers

In [10]:
lr = 2e-5
model_name = "ibm-granite/granite-embedding-english-r2"
output_dir = f"granite-r2-msmarco-{lr}"

model = SentenceTransformer(model_name)

In [11]:
dataset = load_dataset(
        "sentence-transformers/msmarco-co-condenser-margin-mse-sym-mnrl-mean-v1",
        "triplet-hard",
        split="train",
    )
dataset_dict = dataset.train_test_split(test_size=2_000, seed=43)
train_dataset = dataset_dict["train"].select(range(500))
eval_dataset = dataset_dict["test"]

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/17 [00:00<?, ?it/s]

triplet-hard/train-00000-of-00017.parque(…):   0%|          | 0.00/127M [00:00<?, ?B/s]

triplet-hard/train-00001-of-00017.parque(…):   0%|          | 0.00/127M [00:00<?, ?B/s]

triplet-hard/train-00002-of-00017.parque(…):   0%|          | 0.00/127M [00:00<?, ?B/s]

...

triplet-hard/train-00014-of-00017.parque(…):   0%|          | 0.00/127M [00:00<?, ?B/s]

triplet-hard/train-00015-of-00017.parque(…):   0%|          | 0.00/127M [00:00<?, ?B/s]

triplet-hard/train-00016-of-00017.parque(…):   0%|          | 0.00/127M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/11662655 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/17 [00:00<?, ?it/s]

In [12]:
loss = CachedMultipleNegativesRankingLoss(model, mini_batch_size=512)  # Increase mini_batch_size if you have enough VRAM

args = SentenceTransformerTrainingArguments(
    output_dir=output_dir,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=512,
    warmup_ratio=0.05,
    fp16=False,  # Set to False if GPU can't handle FP16
    bf16=True,  # Set to True if GPU supports BF16
    batch_sampler=BatchSamplers.NO_DUPLICATES,  # (Cached)MultipleNegativesRankingLoss benefits from no duplicates
    learning_rate=lr,
    # Optional tracking/debugging parameters:
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    logging_steps=100,
    report_to="none"
)

In [13]:
dev_evaluator = TripletEvaluator(
    anchors=eval_dataset["query"],
    positives=eval_dataset["positive"],
    negatives=eval_dataset["negative"],
    name="msmarco-co-condenser-dev",
    )
dev_evaluator(model)

{'msmarco-co-condenser-dev_cosine_accuracy': 0.9225000143051147}

In [14]:
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    loss=loss,
    evaluator=dev_evaluator,
    )
trainer.train()

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
100,0.255000
200,0.258300


TrainOutput(global_step=250, training_loss=0.25705231857299804, metrics={'train_runtime': 37.9919, 'train_samples_per_second': 13.161, 'train_steps_per_second': 6.58, 'total_flos': 0.0, 'train_loss': 0.25705231857299804, 'epoch': 1.0})

In [15]:
dev_evaluator(model)

{'msmarco-co-condenser-dev_cosine_accuracy': 0.9194999933242798}

In [16]:
model.save_pretrained(f"{output_dir}/final")